In [53]:
import pandas as pd
import numpy as np
import pyarrow
from src import *

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, MinMaxScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import  train_test_split

import keras
from keras.models import Sequential
from keras.layers import Dense, Input, Dropout
from keras import optimizers

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras import regularizers
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.optimizers import Adam
from sklearn.linear_model import LogisticRegression

print(tf.config.list_physical_devices())
BATCH_SIZE = 1024

[PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]


In [50]:
def num_max_neuronio(X, d):
    CT = len(X)
    return int((CT - 10)/(10 * (d + 2)))

# 1.0 - Classificação de Presença Baseada em Fatores Socioeconômicos Usando Redes Neurais

In [2]:
colunas = ['Q001','Q002','Q003', 'Q004', 'Q005', 'Q006', 'Q007', 'Q008', 'Q009', 'Q010', 'Q011', 'Q012', 'Q013', 'Q014', 'Q015', 'Q016', 'Q017', 'Q018', 
           'Q019', 'Q020', 'Q021', 'Q022', 'Q023', 'Q024', 'Q025', 'TP_PRESENCA_LC', 'TP_PRESENCA_CH', 'TP_PRESENCA_CN', 'TP_PRESENCA_MT', 
           'TP_FAIXA_ETARIA', 'TP_SEXO','TP_ESTADO_CIVIL', 'TP_COR_RACA', 'TP_ESCOLA', 'TP_ST_CONCLUSAO', 'IN_TREINEIRO', 
           'NU_ANO', 'TP_LOCALIZACAO_ESC','TP_SIT_FUNC_ESC', 'TP_DEPENDENCIA_ADM_ESC']

df = pd.read_parquet("enem_parquet", columns = colunas)

## 1.1 - Criando uma Amostra

In [3]:
df = df.sample(n=1_000_000, random_state=42)

## 1.2- Limpando os Dados

In [5]:
df = df[df['TP_ESCOLA'].isin([2,3])]
df = df[df['TP_ESTADO_CIVIL'].isin([1,2,3,4])]

In [6]:
df['TP_LOCALIZACAO_ESC'] = df['TP_LOCALIZACAO_ESC'].fillna(0)
df['TP_DEPENDENCIA_ADM_ESC'] = df['TP_DEPENDENCIA_ADM_ESC'].fillna(0)
df['TP_SIT_FUNC_ESC'] = df['TP_SIT_FUNC_ESC'].fillna(0)

In [8]:
questionario = [f'Q{str(i).zfill(3)}' for i in range(1, 26) if i != 5]
df = df.dropna(subset=questionario)

## 1.3 - Criando Rótulo

In [4]:
FALTOU = (
    (df['TP_PRESENCA_CH'] != 1) | 
    (df['TP_PRESENCA_LC'] != 1) | 
    (df['TP_PRESENCA_CN'] != 1) | 
    (df['TP_PRESENCA_MT'] != 1)
)

df['FALTOU'] = FALTOU.astype(int)

## 1.4 - Tratando os Dados

In [9]:
nominais = ['TP_ESCOLA', 'TP_DEPENDENCIA_ADM_ESC',
            'TP_ESTADO_CIVIL', 'TP_LOCALIZACAO_ESC', 'TP_SIT_FUNC_ESC']

ordinais = ['TP_FAIXA_ETARIA', 'TP_ST_CONCLUSAO']

categorias_quest = []
for i in range(1, 26):
    if i == 5:
        continue
    elif i == 6:
        categorias_quest.append(list('ABCDEFGHIJKLMNOPQ'))  
    elif i == 25:
        categorias_quest.append(list('AB'))                  
    elif i in [1, 2]:
        categorias_quest.append(list('ABCDEFGH'))            
    elif i in [3, 4]:
        categorias_quest.append(list('ABCDEF'))              
    else:
        categorias_quest.append(list('ABCDE'))               

binarias = ['IN_TREINEIRO']

In [10]:
pipe_nominal = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

pipe_ordinal = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)

pipe_quest = OrdinalEncoder(
    categories=categorias_quest,
    handle_unknown='use_encoded_value',
    unknown_value=-1
)

pipe_continua = MinMaxScaler()

preprocessor = ColumnTransformer(transformers=[
    ('nominal',      pipe_nominal,  nominais),
    ('ordinal',      pipe_ordinal,  ordinais),
    ('questionario', pipe_quest,    questionario),
    ('binaria',      'passthrough', binarias),
], remainder='drop')

## 1.5- Construção da Matriz X e Vetor y

In [11]:
X = preprocessor.fit_transform(df[colunas])
X = X.astype(np.float32)
y = df['FALTOU']

## 1.6 - Separação em Dados de Treino e Teste

In [12]:
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
x_train, x_val, y_train, y_val = train_test_split(x_train, y_train, test_size=0.2, random_state=42)

## 1.7 - Computando Peso das Classes 

In [22]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

class_weights = dict(enumerate(weights))

## 1.8 - Treinando a Rede Neural

In [14]:
max_neurons = num_max_neuronio(x_train, d = 1)
print("Número máximo de neurônios:", max_neurons)

Número máximo de neurônios: 6361


In [23]:
model = Sequential()

# model.add(Dense(256, input_dim=x_train.shape[1], kernel_initializer='he_normal', activation='relu'))
# model.add(BatchNormalization())
# model.add(Dropout(0.2))

model.add(Dense(100, input_dim=x_train.shape[1], kernel_initializer='he_normal', activation='relu'))
# model.add(BatchNormalization())
# model.add(Dropout(0.2))

# model.add(Dense(64, kernel_initializer='he_normal', activation='relu'))
# model.add(BatchNormalization())
# model.add(Dropout(0.2))

model.add(Dense(1, activation='sigmoid'))

model.compile(loss='binary_crossentropy', optimizer=Adam(), metrics=['accuracy'])


es = EarlyStopping(monitor='val_loss', mode='min', patience=10, verbose=1)

model.fit(
    x_train,
    y_train,
    validation_data=(x_val, y_val),
    epochs=100,
    batch_size=128,
    callbacks=[es],
    class_weight=class_weights
)

Epoch 1/100


C:\Users\Windows 11\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


1492/1492 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.6224 - loss: 0.6324 - val_accuracy: 0.6784 - val_loss: 0.5862
Epoch 2/100
1492/1492 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.6310 - loss: 0.6248 - val_accuracy: 0.6113 - val_loss: 0.6336
Epoch 3/100
1492/1492 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.6322 - loss: 0.6226 - val_accuracy: 0.5879 - val_loss: 0.6590
Epoch 4/100
1492/1492 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.6339 - loss: 0.6212 - val_accuracy: 0.6177 - val_loss: 0.6366
Epoch 5/100
1492/1492 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.6375 - loss: 0.6201 - val_accuracy: 0.6541 - val_loss: 0.6076
Epoch 6/100
1492/1492 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.6374 - loss: 0.6194 - val_accuracy: 0.6442 - val_loss: 0.6111
Epoch 7/100
1492/1492 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.6378 - loss: 0.6188 - val_accuracy: 0.6763 - val_loss: 0.5912
Epoch 8/100
1492/1492 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.6389 - loss: 0.6183 - val_

## 1.9 - Exibindo Resultados no Teste

In [24]:
y_pred = (model.predict(x_test) > 0.5).astype(int).flatten()
print(classification_report(y_test, y_pred))

1864/1864 ━━━━━━━━━━━━━━━━━━━━ 2s 957us/step
              precision    recall  f1-score   support

           0       0.84      0.62      0.71     43713
           1       0.40      0.68      0.50     15932

    accuracy                           0.64     59645
   macro avg       0.62      0.65      0.61     59645
weighted avg       0.72      0.64      0.66     59645



## Seção de Testes

In [47]:
y_val_prob = model.predict(x_val).flatten()

calibrator = LogisticRegression()

calibrator.fit(
    y_val_prob.reshape(-1,1),
    y_val
)

x_aluno = pd.DataFrame([{

    # Questionário socioeconômico
    'Q001': np.random.choice(list('ABCDEFGH')),
    'Q002': np.random.choice(list('ABCDEFGH')),
    'Q003': np.random.choice(list('ABCDEF')),
    'Q004': np.random.choice(list('ABCDEF')),
    'Q006': np.random.choice(list('ABCDEFGHIJKLMNOPQ')),
    'Q007': np.random.choice(list('ABCD')),
    'Q008': np.random.choice(list('ABCDE')),
    'Q009': np.random.choice(list('ABCDE')),
    'Q010': np.random.choice(list('ABCDE')),
    'Q011': np.random.choice(list('ABCDE')),
    'Q012': np.random.choice(list('ABCDE')),
    'Q013': np.random.choice(list('ABCDE')),
    'Q014': np.random.choice(list('ABCDE')),
    'Q015': np.random.choice(list('ABCDE')),
    'Q016': np.random.choice(list('ABCDE')),
    'Q017': np.random.choice(list('ABCDE')),
    'Q018': np.random.choice(list('ABCDE')),
    'Q019': np.random.choice(list('ABCDE')),
    'Q020': np.random.choice(list('ABCDE')),
    'Q021': np.random.choice(list('ABCDE')),
    'Q022': np.random.choice(list('ABCDE')),
    'Q023': np.random.choice(list('ABCDE')),
    'Q024': np.random.choice(list('ABCDE')),
    'Q025': np.random.choice(list('AB')),

    # Categóricas
    'TP_FAIXA_ETARIA': np.random.choice([3,4,2,1,14,6,5,12,8,7,11,9,13,16,10,17,15,19,18,20]),
    'TP_ST_CONCLUSAO': 2,
    'TP_ESCOLA': np.random.choice([2,3]),
    'TP_DEPENDENCIA_ADM_ESC': np.random.choice([0.,4.,2.,1.,3.]),
    'TP_ESTADO_CIVIL': np.random.choice([1,2,3,4]),
    'TP_LOCALIZACAO_ESC': np.random.choice([0.,1.,2.]),
    'TP_SIT_FUNC_ESC': np.random.choice([0.,1.,3.,4.,2.]),

    # Binária
    'IN_TREINEIRO': np.random.choice([0,1]),

}])

x_aluno = pd.DataFrame([{

    # Questionário socioeconômico
    'Q001': np.random.choice(list('ABCDEFGH')),
    'Q002': np.random.choice(list('ABCDEFGH')),
    'Q003': np.random.choice(list('ABCDEF')),
    'Q004': np.random.choice(list('ABCDEF')),
    'Q006': np.random.choice(list('ABCDEFGHIJKLMNOPQ')),
    'Q007': np.random.choice(list('ABCD')),
    'Q008': np.random.choice(list('ABCDE')),
    'Q009': np.random.choice(list('ABCDE')),
    'Q010': np.random.choice(list('ABCDE')),
    'Q011': np.random.choice(list('ABCDE')),
    'Q012': np.random.choice(list('ABCDE')),
    'Q013': np.random.choice(list('ABCDE')),
    'Q014': np.random.choice(list('ABCDE')),
    'Q015': np.random.choice(list('ABCDE')),
    'Q016': np.random.choice(list('ABCDE')),
    'Q017': np.random.choice(list('ABCDE')),
    'Q018': np.random.choice(list('ABCDE')),
    'Q019': np.random.choice(list('ABCDE')),
    'Q020': np.random.choice(list('ABCDE')),
    'Q021': np.random.choice(list('ABCDE')),
    'Q022': np.random.choice(list('ABCDE')),
    'Q023': np.random.choice(list('ABCDE')),
    'Q024': np.random.choice(list('ABCDE')),
    'Q025': np.random.choice(list('AB')),

    # Categóricas
    'TP_FAIXA_ETARIA': np.random.choice([3,4,2,1,14,6,5,12,8,7,11,9,13,16,10,17,15,19,18,20]),
    'TP_ST_CONCLUSAO': 2,
    'TP_ESCOLA': np.random.choice([2,3]),
    'TP_DEPENDENCIA_ADM_ESC': np.random.choice([0.,4.,2.,1.,3.]),
    'TP_ESTADO_CIVIL': np.random.choice([1,2,3,4]),
    'TP_LOCALIZACAO_ESC': np.random.choice([0.,1.,2.]),
    'TP_SIT_FUNC_ESC': np.random.choice([0.,1.,3.,4.,2.]),

    # Binária
    'IN_TREINEIRO': np.random.choice([0,1]),

}])

x_caio = pd.DataFrame([{

'Q001':'H',
'Q002':'G',
'Q003':'D',
'Q004':'D',
'Q006':'F',
'Q007':'C',
'Q008':'D',
'Q009':'D',
'Q010':'B',
'Q011':'E',
'Q012':'D',
'Q013':'C',
'Q014':'C',
'Q015':'E',
'Q016':'D',
'Q017':'C',
'Q018':'C',
'Q019':'D',
'Q020':'C',
'Q021':'D',
'Q022':'C',
'Q023':'C',
'Q024':'D',
'Q025':'B',

'TP_FAIXA_ETARIA':5,
'TP_ST_CONCLUSAO':2,
'TP_ESCOLA':3,
'TP_DEPENDENCIA_ADM_ESC':4.0,
'TP_ESTADO_CIVIL':1,
'TP_LOCALIZACAO_ESC':1.0,
'TP_SIT_FUNC_ESC':1.0,

'IN_TREINEIRO':0

}])

x_aluno_transformado = preprocessor.transform(x_caio).astype(np.float32)

prob_raw = model.predict(x_aluno_transformado).flatten()[0]

prob_calibrada = calibrator.predict_proba([[prob_raw]])[0][1]

x_aluno_resultado = x_caio.copy()

x_aluno_resultado['Prob_Faltar_Bruta'] = prob_raw
x_aluno_resultado['Prob_Faltar_Calibrada'] = prob_calibrada

print(f"Probabilidade (modelo): {prob_raw:.1%}")
print(f"Probabilidade (calibrada): {prob_calibrada:.1%}")

print("\nPerfil do aluno:")
print(x_aluno_resultado.T)

1492/1492 ━━━━━━━━━━━━━━━━━━━━ 1s 986us/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
Probabilidade (modelo): 40.6%
Probabilidade (calibrada): 19.8%

Perfil do aluno:
                               0
Q001                           H
Q002                           G
Q003                           D
Q004                           D
Q006                           F
Q007                           C
Q008                           D
Q009                           D
Q010                           B
Q011                           E
Q012                           D
Q013                           C
Q014                           C
Q015                           E
Q016                           D
Q017                           C
Q018                           C
Q019                           D
Q020                           C
Q021                           D
Q022                           C
Q023                           C
Q024                           D
Q025                           B
TP_FAIXA_ET